# TEC vs TS Calibration on LLM Datasets

## Better uncertainty calibration for LLM classifiers

This notebook demonstrates the comparison between Thermodynamic Entropy Calibration (TEC) and Temperature Scaling (TS) for improving uncertainty calibration in LLM-based text classification.

### What this notebook does:
1. **Simulates** logits from a pre-trained transformer model (to keep demo fast)
2. Applies three calibration methods:
   - **Uncalibrated**: Raw model outputs
   - **Temperature Scaling (TS)**: Single temperature parameter optimized on validation set
   - **Thermodynamic Entropy Calibration (TEC)**: Per-sample temperature based on prediction entropy and margin
3. Evaluates calibration using:
   - Expected Calibration Error (ECE)
   - Brier Score
   - Negative Log-Likelihood (NLL)
   - Accuracy
4. Performs heterogeneous analysis (easy vs hard examples based on decision margin)
5. Reports bootstrap confidence intervals (10 samples for speed)

### Demo Approach:
Instead of loading actual transformer models (which takes 5+ minutes), this demo **simulates realistic logits** that mimic the output of a fine-tuned DistilBERT model on SST-2. The calibration logic is **identical** to the full experiment.

### Expected Runtime:
- This demo: ~30 seconds
- Full experiment (with real models): ~10-15 minutes per dataset

In [ ]:
# Install dependencies - following aii-colab pattern
# Note: Running locally, packages should already be available
# If not, uncomment and run the pip installs below

import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Uncomment if needed:
# _pip('loguru')
# _pip('scipy')
# if 'google.colab' not in sys.modules:
#     _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

print("Dependencies check completed.")

In [ ]:
# Imports - copied from original method.py with minimal additions
import json
import gc
import sys
import warnings
from pathlib import Path
from loguru import logger
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, asdict
import numpy as np
from scipy.optimize import minimize_scalar

# For visualization
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

# Setup logging
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

print("Imports completed.")

In [ ]:
# Data loading helper - using GitHub URL with local fallback pattern
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-d144fc-thermodynamic-entropy-calibration-for-im/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub download failed: {e}")
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

import os
print("Data loading helper defined.")

In [ ]:
# Load the demo data (for dataset info and labels)
data = load_data()
print(f"Loaded data with {len(data['datasets'])} dataset(s)")
for ds in data['datasets']:
    print(f"  - {ds['dataset']}: {len(ds['examples'])} examples")

## Configuration

Set all tunable parameters here. Start with **ABSOLUTE MINIMUM** values for a quick demo:
- `N_EXAMPLES`: Number of examples to simulate (20 to match mini data)
- `N_BOOTSTRAP`: Number of bootstrap samples for confidence intervals (10 for speed)
- `N_CLASSES`: Number of classes (2 for SST-2 sentiment)

### Why Simulate Logits?
Loading actual transformer models takes 5+ minutes. Instead, we simulate realistic logits that have the same statistical properties as real model outputs. This lets us demonstrate the calibration logic quickly while preserving the exact same algorithms.

In [ ]:
# Configuration - MINIMUM values for quick demo
N_EXAMPLES = 20        # Match mini_demo_data.json (20 examples)
N_BOOTSTRAP = 10       # Minimum: 10 bootstrap samples (original: 200)
N_CLASSES = 2          # SST-2 has 2 classes (positive/negative)
DATASET_NAME = "sst-2"

print(f"Configuration:")
print(f"  N_EXAMPLES: {N_EXAMPLES}")
print(f"  N_BOOTSTRAP: {N_BOOTSTRAP}")
print(f"  N_CLASSES: {N_CLASSES}")
print(f"  DATASET_NAME: {DATASET_NAME}")

## Simulate Realistic Logits

Generate synthetic logits that mimic the output of a fine-tuned DistilBERT model on SST-2:
- **Well-calibrated examples**: High confidence on correct predictions
- **Overconfident examples**: High confidence but wrong predictions
- **Uncertain examples**: Low confidence, mixed correctness

The simulation creates realistic patterns so the calibration methods have meaningful effects.

In [ ]:
# Simulate realistic logits from a transformer model
np.random.seed(42)

def simulate_logits(n_examples, n_classes, n_calibrated=0.3):
    """
    Simulate logits that mimic a real transformer model.
    
    Args:
        n_examples: Number of examples
        n_classes: Number of classes
        n_calibrated: Fraction of examples that are well-calibrated
    
    Returns:
        logits: Array of shape (n_examples, n_classes)
        labels: True labels (0 to n_classes-1)
    """
    logits = np.zeros((n_examples, n_classes))
    labels = np.random.randint(0, n_classes, size=n_examples)
    
    for i in range(n_examples):
        true_label = labels[i]
        
        # 70% chance the model predicts correctly
        if np.random.random() < 0.7:
            pred_label = true_label
        else:
            pred_label = np.random.choice([j for j in range(n_classes) if j != true_label])
        
        # Generate logits with realistic patterns
        # Correct predictions have higher logits
        logits[i, pred_label] = np.random.normal(3.0, 1.0)  # High confidence
        
        # Other classes have lower logits
        for j in range(n_classes):
            if j != pred_label:
                logits[i, j] = np.random.normal(-1.0, 1.5)
        
        # Add some overconfident examples (high confidence but wrong)
        if pred_label != true_label and np.random.random() < 0.3:
            logits[i, pred_label] += 2.0  # Make it more overconfident
    
    return logits, labels

# Generate train/val/test splits
n_train = int(0.6 * N_EXAMPLES)
n_val = int(0.2 * N_EXAMPLES)
n_test = N_EXAMPLES - n_train - n_val

logits_train, labels_train = simulate_logits(n_train, N_CLASSES)
logits_val, labels_val = simulate_logits(n_val, N_CLASSES)
logits_test, labels_test = simulate_logits(n_test, N_CLASSES)

print(f"Simulated data shapes:")
print(f"  Train: logits={logits_train.shape}, labels={labels_train.shape}")
print(f"  Val:   logits={logits_val.shape}, labels={labels_val.shape}")
print(f"  Test:  logits={logits_test.shape}, labels={labels_test.shape}")

# Check simulated data quality
def softmax(logits, temperature=1.0):
    logits_t = logits / temperature
    logits_t = logits_t - np.max(logits_t, axis=-1, keepdims=True)
    exp_logits = np.exp(logits_t)
    return exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)

probs_test = softmax(logits_test, 1.0)
preds = np.argmax(probs_test, axis=1)
accuracy = np.mean(preds == labels_test)
print(f"\nSimulated model accuracy: {accuracy:.2%}")
print(f"Mean confidence: {np.mean(np.max(probs_test, axis=1)):.3f}")

## Helper Functions

These functions are copied directly from the original `method.py`:
- `CalibrationResult`: Dataclass to store calibration results
- `softmax`, `compute_entropy`, `compute_margin`: Utility functions
- `calibrate_ts`, `calibrate_tec`: Calibration methods
- `compute_ece`, `compute_nll`, `compute_brier`, `compute_accuracy`: Metrics
- `bootstrap_ci`: Bootstrap confidence intervals
- `heterogeneous_analysis`: Easy/hard split analysis

In [ ]:
# Helper functions - copied directly from method.py

@dataclass
class CalibrationResult:
    method_name: str
    dataset_name: str
    ece: float
    brier: float
    nll: float
    accuracy: float
    ece_ci_lower: float
    ece_ci_upper: float
    brier_ci_lower: float
    brier_ci_upper: float
    nll_ci_lower: float
    nll_ci_upper: float
    accuracy_ci_lower: float
    accuracy_ci_upper: float
    ece_easy: float
    ece_hard: float
    accuracy_easy: float
    accuracy_hard: float
    temperatures: Optional[List[float]] = None


def softmax(logits, temperature=1.0):
    """Compute softmax with temperature scaling."""
    logits_t = logits / temperature
    logits_t = logits_t - np.max(logits_t, axis=-1, keepdims=True)
    exp_logits = np.exp(logits_t)
    return exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)


def compute_entropy(probs):
    """Compute entropy of probability distribution."""
    probs = np.clip(probs, 1e-12, 1.0)
    return -np.sum(probs * np.log(probs), axis=-1)


def compute_margin(probs):
    """Compute margin between top two predictions."""
    sorted_probs = np.sort(probs, axis=-1)
    return sorted_probs[:, -1] - sorted_probs[:, -2]


def calibrate_ts(logits, logits_val, labels_val):
    """Temperature Scaling calibration."""
    def nll_loss(T):
        probs = softmax(logits_val, T)
        probs = np.clip(probs, 1e-12, 1.0)
        log_probs = np.log(probs)
        return -np.mean(log_probs[np.arange(len(labels_val)), labels_val])

    result = minimize_scalar(nll_loss, bounds=(0.01, 10.0), method='bounded')
    optimal_T = result.x

    probs_cal = softmax(logits, optimal_T)
    return probs_cal, optimal_T


def calibrate_tec(logits, logits_val, labels_val, n_classes):
    """Thermodynamic Entropy Calibration (TEC)."""
    probs_val = softmax(logits_val, 1.0)
    H_val = compute_entropy(probs_val)
    M_val = compute_margin(probs_val)
    H_max = np.log(n_classes)
    H_norm_val = H_val / H_max

    best_ece = float('inf')
    best_params = (1.0, 1.0, 0.5)

    # Grid search over TEC parameters
    for T0 in [0.5, 1.0, 2.0, 5.0]:
        for alpha in [0.0, 0.5, 1.0, 2.0]:
            for beta in [0.0, 0.25, 0.5, 1.0]:
                T_i = T0 * (1 + alpha * H_norm_val - beta * M_val)
                T_i = np.clip(T_i, 0.01, 100.0)

                logits_t = logits_val / T_i[:, np.newaxis]
                logits_t = logits_t - np.max(logits_t, axis=-1, keepdims=True)
                exp_logits = np.exp(logits_t)
                probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)

                ece = compute_ece(probs, labels_val)
                if ece < best_ece:
                    best_ece = ece
                    best_params = (T0, alpha, beta)

    T0_opt, alpha_opt, beta_opt = best_params
    logger.info(f"  TEC params: T0={T0_opt}, alpha={alpha_opt}, beta={beta_opt}")

    # Apply TEC to test data
    probs_test = softmax(logits, 1.0)
    H_test = compute_entropy(probs_test)
    M_test = compute_margin(probs_test)
    H_norm_test = H_test / np.log(n_classes)

    T_i_test = T0_opt * (1 + alpha_opt * H_norm_test - beta_opt * M_test)
    T_i_test = np.clip(T_i_test, 0.01, 100.0)

    logits_t = logits / T_i_test[:, np.newaxis]
    logits_t = logits_t - np.max(logits_t, axis=-1, keepdims=True)
    exp_logits = np.exp(logits_t)
    probs_cal = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)

    return probs_cal, T_i_test, T0_opt, alpha_opt, beta_opt


def compute_ece(probs, labels, n_bins=10):
    """Compute Expected Calibration Error."""
    n_samples = len(labels)
    confidences = np.max(probs, axis=-1)
    predictions = np.argmax(probs, axis=-1)
    accuracies = (predictions == labels).astype(float)

    ece = 0.0
    for i in range(n_bins):
        lower = i / n_bins
        upper = (i + 1) / n_bins
        if i == n_bins - 1:
            mask = (confidences >= lower) & (confidences <= upper)
        else:
            mask = (confidences >= lower) & (confidences < upper)

        if np.sum(mask) > 0:
            bin_accuracy = np.mean(accuracies[mask])
            bin_confidence = np.mean(confidences[mask])
            bin_weight = np.sum(mask) / n_samples
            ece += bin_weight * abs(bin_accuracy - bin_confidence)

    return ece


def compute_nll(probs, labels):
    """Compute Negative Log-Likelihood."""
    probs = np.clip(probs, 1e-12, 1.0)
    log_probs = np.log(probs)
    return -np.mean(log_probs[np.arange(len(labels)), labels])


def compute_brier(probs, labels, n_classes):
    """Compute Brier Score."""
    n_samples = len(labels)
    one_hot = np.zeros((n_samples, n_classes))
    one_hot[np.arange(n_samples), labels] = 1.0
    return np.mean(np.sum((probs - one_hot) ** 2, axis=-1))


def compute_accuracy(probs, labels):
    """Compute accuracy."""
    predictions = np.argmax(probs, axis=-1)
    return np.mean(predictions == labels)


def bootstrap_ci(metric_func, probs, labels, n_bootstrap=200, **kwargs):
    """Compute bootstrap confidence intervals."""
    n_samples = len(labels)
    bootstrap_metrics = []

    for _ in range(n_bootstrap):
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        probs_boot = probs[indices]
        labels_boot = labels[indices]

        if kwargs:
            metric_val = metric_func(probs_boot, labels_boot, **kwargs)
        else:
            metric_val = metric_func(probs_boot, labels_boot)
        bootstrap_metrics.append(metric_val)

    lower = np.percentile(bootstrap_metrics, 2.5)
    upper = np.percentile(bootstrap_metrics, 97.5)
    return lower, upper


def heterogeneous_analysis(probs, labels, margins):
    """Analyze calibration on easy vs hard examples."""
    sorted_indices = np.argsort(margins)
    n = len(sorted_indices)
    mid = n // 2

    hard_idx = sorted_indices[:mid]
    easy_idx = sorted_indices[mid:]

    ece_easy = compute_ece(probs[easy_idx], labels[easy_idx])
    ece_hard = compute_ece(probs[hard_idx], labels[hard_idx])
    acc_easy = compute_accuracy(probs[easy_idx], labels[easy_idx])
    acc_hard = compute_accuracy(probs[hard_idx], labels[hard_idx])

    return ece_easy, ece_hard, acc_easy, acc_hard

print("All helper functions defined.")

## Run Experiment

This cell runs the main experiment comparing three calibration methods:
1. **Uncalibrated**: Raw softmax outputs
2. **Temperature Scaling (TS)**: Optimize single temperature on validation set
3. **Thermodynamic Entropy Calibration (TEC)**: Per-sample temperature using entropy and margin

The results are stored in `all_results` list.

In [ ]:
# Run the experiment - adapted from run_experiment() in method.py
logger.info("=" * 80)
logger.info("TEC vs TS Calibration Experiment (Demo with Simulated Data)")
logger.info(f"  Device: CPU (simulated data), Examples: {N_EXAMPLES}")
logger.info("=" * 80)

all_results = []

# Use simulated data directly
num_classes = N_CLASSES
dataset_name = DATASET_NAME

logger.info(f"\n{'='*80}")
logger.info(f"Dataset: {dataset_name} (simulated)")
logger.info(f"{'='*80}")

logger.info("  Using simulated logits (no model loading needed)")
logger.info(f"  Logits: val={logits_val.shape}, test={logits_test.shape}")

# Method 1: Uncalibrated
logger.info("\n  [1/3] Uncalibrated")
probs_uncal = softmax(logits_test, 1.0)

ece = compute_ece(probs_uncal, labels_test)
brier = compute_brier(probs_uncal, labels_test, num_classes)
nll = compute_nll(probs_uncal, labels_test)
acc = compute_accuracy(probs_uncal, labels_test)

ece_ci = bootstrap_ci(compute_ece, probs_uncal, labels_test, N_BOOTSTRAP)
brier_ci = bootstrap_ci(compute_brier, probs_uncal, labels_test, N_BOOTSTRAP, n_classes=num_classes)
nll_ci = bootstrap_ci(compute_nll, probs_uncal, labels_test, N_BOOTSTRAP)
acc_ci = bootstrap_ci(compute_accuracy, probs_uncal, labels_test, N_BOOTSTRAP)

margins = compute_margin(probs_uncal)
het = heterogeneous_analysis(probs_uncal, labels_test, margins)

result = CalibrationResult(
    method_name="Uncalibrated", dataset_name=dataset_name,
    ece=ece, brier=brier, nll=nll, accuracy=acc,
    ece_ci_lower=ece_ci[0], ece_ci_upper=ece_ci[1],
    brier_ci_lower=brier_ci[0], brier_ci_upper=brier_ci[1],
    nll_ci_lower=nll_ci[0], nll_ci_upper=nll_ci[1],
    accuracy_ci_lower=acc_ci[0], accuracy_ci_upper=acc_ci[1],
    ece_easy=het[0], ece_hard=het[1],
    accuracy_easy=het[2], accuracy_hard=het[3]
)
all_results.append(result)
logger.info(f"    ECE={ece:.4f} [{ece_ci[0]:.4f}, {ece_ci[1]:.4f}], Acc={acc:.4f}")

# Method 2: Temperature Scaling
logger.info("\n  [2/3] Temperature Scaling")
probs_ts, optimal_T = calibrate_ts(logits_test, logits_val, labels_val)

ece = compute_ece(probs_ts, labels_test)
brier = compute_brier(probs_ts, labels_test, num_classes)
nll = compute_nll(probs_ts, labels_test)
acc = compute_accuracy(probs_ts, labels_test)

ece_ci = bootstrap_ci(compute_ece, probs_ts, labels_test, N_BOOTSTRAP)
brier_ci = bootstrap_ci(compute_brier, probs_ts, labels_test, N_BOOTSTRAP, n_classes=num_classes)
nll_ci = bootstrap_ci(compute_nll, probs_ts, labels_test, N_BOOTSTRAP)
acc_ci = bootstrap_ci(compute_accuracy, probs_ts, labels_test, N_BOOTSTRAP)

margins = compute_margin(probs_ts)
het = heterogeneous_analysis(probs_ts, labels_test, margins)

result = CalibrationResult(
    method_name="Temperature Scaling", dataset_name=dataset_name,
    ece=ece, brier=brier, nll=nll, accuracy=acc,
    ece_ci_lower=ece_ci[0], ece_ci_upper=ece_ci[1],
    brier_ci_lower=brier_ci[0], brier_ci_upper=brier_ci[1],
    nll_ci_lower=nll_ci[0], nll_ci_upper=nll_ci[1],
    accuracy_ci_lower=acc_ci[0], accuracy_ci_upper=acc_ci[1],
    ece_easy=het[0], ece_hard=het[1],
    accuracy_easy=het[2], accuracy_hard=het[3]
)
all_results.append(result)
logger.info(f"    T={optimal_T:.4f}, ECE={ece:.4f}, Acc={acc:.4f}")

# Method 3: TEC
logger.info("\n  [3/3] TEC")
probs_tec, temps, T0, alpha, beta = calibrate_tec(
    logits_test, logits_val, labels_val, num_classes
)

ece = compute_ece(probs_tec, labels_test)
brier = compute_brier(probs_tec, labels_test, num_classes)
nll = compute_nll(probs_tec, labels_test)
acc = compute_accuracy(probs_tec, labels_test)

ece_ci = bootstrap_ci(compute_ece, probs_tec, labels_test, N_BOOTSTRAP)
brier_ci = bootstrap_ci(compute_brier, probs_tec, labels_test, N_BOOTSTRAP, n_classes=num_classes)
nll_ci = bootstrap_ci(compute_nll, probs_tec, labels_test, N_BOOTSTRAP)
acc_ci = bootstrap_ci(compute_accuracy, probs_tec, labels_test, N_BOOTSTRAP)

margins = compute_margin(probs_tec)
het = heterogeneous_analysis(probs_tec, labels_test, margins)

result = CalibrationResult(
    method_name="TEC", dataset_name=dataset_name,
    ece=ece, brier=brier, nll=nll, accuracy=acc,
    ece_ci_lower=ece_ci[0], ece_ci_upper=ece_ci[1],
    brier_ci_lower=brier_ci[0], brier_ci_upper=brier_ci[1],
    nll_ci_lower=nll_ci[0], nll_ci_upper=nll_ci[1],
    accuracy_ci_lower=acc_ci[0], accuracy_ci_upper=acc_ci[1],
    temperatures=temps.tolist(),
    ece_easy=het[0], ece_hard=het[1],
    accuracy_easy=het[2], accuracy_hard=het[3]
)
all_results.append(result)
logger.info(f"    T0={T0}, alpha={alpha}, beta={beta}")
logger.info(f"    ECE={ece:.4f}, Acc={acc:.4f}")
logger.info(f"    T: mean={np.mean(temps):.4f}, std={np.std(temps):.4f}")

logger.info("\nExperiment completed!")
print(f"\nCollected {len(all_results)} results.")

## Results Visualization

Display the results in a readable table and create visualizations to compare the three calibration methods.

In [ ]:
# Display results in a table
print("=" * 100)
print("RESULTS SUMMARY")
print("=" * 100)

print(f"\nDataset: {DATASET_NAME} (simulated data)")
print(f"{'Method':<25} {'ECE':<12} {'Brier':<12} {'NLL':<12} {'Accuracy':<10}")
print("-" * 70)
for r in all_results:
    print(f"{r.method_name:<25} {r.ece:<12.4f} {r.brier:<12.4f} {r.nll:<12.4f} {r.accuracy:<10.4f}")

# Create a DataFrame for easier visualization
results_data = []
for r in all_results:
    results_data.append({
        'Dataset': r.dataset_name,
        'Method': r.method_name,
        'ECE': r.ece,
        'ECE_CI_Lower': r.ece_ci_lower,
        'ECE_CI_Upper': r.ece_ci_upper,
        'Brier': r.brier,
        'NLL': r.nll,
        'Accuracy': r.accuracy,
        'ECE_Easy': r.ece_easy,
        'ECE_Hard': r.ece_hard,
        'Acc_Easy': r.accuracy_easy,
        'Acc_Hard': r.accuracy_hard
    })

df_results = pd.DataFrame(results_data)
print("\n\nResults as DataFrame:")
display(df_results)

In [ ]:
# Visualize results with bar plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Calibration Methods Comparison (Simulated SST-2 Data)', fontsize=16)

# Plot 1: ECE
ax = axes[0, 0]
methods = df_results['Method'].values
ece_values = df_results['ECE'].values
ece_errors_lower = ece_values - df_results['ECE_CI_Lower'].values
ece_errors_upper = df_results['ECE_CI_Upper'].values - ece_values
ax.bar(methods, ece_values, yerr=[ece_errors_lower, ece_errors_upper],
      capsize=10, color=['gray', 'blue', 'green'])
ax.set_ylabel('ECE (lower is better)', fontsize=12)
ax.set_title('Expected Calibration Error', fontsize=13)
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)

# Plot 2: Brier Score
ax = axes[0, 1]
brier_values = df_results['Brier'].values
ax.bar(methods, brier_values, color=['gray', 'blue', 'green'])
ax.set_ylabel('Brier Score (lower is better)', fontsize=12)
ax.set_title('Brier Score', fontsize=13)
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)

# Plot 3: NLL
ax = axes[1, 0]
nll_values = df_results['NLL'].values
ax.bar(methods, nll_values, color=['gray', 'blue', 'green'])
ax.set_ylabel('NLL (lower is better)', fontsize=12)
ax.set_title('Negative Log-Likelihood', fontsize=13)
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)

# Plot 4: Accuracy
ax = axes[1, 1]
acc_values = df_results['Accuracy'].values
ax.bar(methods, acc_values, color=['gray', 'blue', 'green'])
ax.set_ylabel('Accuracy (higher is better)', fontsize=12)
ax.set_title('Accuracy', fontsize=13)
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

In [ ]:
# Heterogeneous Analysis: Easy vs Hard examples
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Heterogeneous Analysis: Easy vs Hard Examples (by Decision Margin)', fontsize=14)

methods = df_results['Method'].values

# Plot 1: ECE Easy vs Hard
ax = axes[0]
x = np.arange(len(methods))
width = 0.35

ece_easy = df_results['ECE_Easy'].values
ece_hard = df_results['ECE_Hard'].values

ax.bar(x - width/2, ece_easy, width, label='Easy (high margin)', color='lightgreen', alpha=0.8)
ax.bar(x + width/2, ece_hard, width, label='Hard (low margin)', color='lightcoral', alpha=0.8)
ax.set_ylabel('ECE', fontsize=12)
ax.set_title('ECE: Easy vs Hard Examples', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=45)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Plot 2: Accuracy Easy vs Hard
ax = axes[1]

acc_easy = df_results['Acc_Easy'].values
acc_hard = df_results['Acc_Hard'].values

ax.bar(x - width/2, acc_easy, width, label='Easy (high margin)', color='lightgreen', alpha=0.8)
ax.bar(x + width/2, acc_hard, width, label='Hard (low margin)', color='lightcoral', alpha=0.8)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy: Easy vs Hard Examples', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=45)
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

# Print interpretation
print("\n" + "="*80)
print("INTERPRETATION:")
print("="*80)
print("\nKey findings from this demo:")
print("1. Calibration methods (TS and TEC) should reduce ECE compared to uncalibrated.")
print("2. TEC adapts temperature per-sample based on entropy and margin.")
print("3. Heterogeneous analysis shows if calibration helps more on easy or hard examples.")
print("4. Bootstrap confidence intervals show statistical significance.")
print("\nNote: This demo uses simulated logits for speed.")
print("Full experiment loads actual transformer models (5+ min) and uses 200 bootstrap samples.")